# Guitar MIDI — pipeline Kaggle

Ce notebook clone une branche d'expérience, monte un dataset privé attaché et lance soit le smoke test, soit le train complet, soit la reconstruction de `data/processed`. Le package de train est refusé s'il contient le split test.

In [ ]:
TASK = "smoke"  # "smoke", "train" ou "rebuild"
BRANCH = "codex/cleanup-cloud-training-docs"
REPO_URL = "https://github.com/Andriamarosoa/midi.git"
WORKSPACE = "/kaggle/working/midi"
WORKERS = 4


In [ ]:
import importlib
import json
import os
import pathlib
import shutil
import subprocess
import sys
import tarfile
import time

if not ((3, 9) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(f"Python incompatible: {sys.version.split()[0]}")
workspace = pathlib.Path(WORKSPACE)
if workspace.exists():
    shutil.rmtree(workspace)
source_archives = sorted(pathlib.Path("/kaggle/input").rglob("midi_source.tar.gz"))
if source_archives:
    if len(source_archives) != 1:
        raise RuntimeError(f"Expected one source archive, got {source_archives}")
    metadata_path = source_archives[0].with_name("source_metadata.json")
    source_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    workspace.mkdir(parents=True)
    with tarfile.open(source_archives[0], "r:gz") as archive:
        destination = workspace.resolve()
        for member in archive.getmembers():
            target = (workspace / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Source archive escapes workspace: {member.name}")
        archive.extractall(workspace)
    os.environ["GUITAR_MIDI_SOURCE_BRANCH"] = source_metadata["branch"]
    os.environ["GUITAR_MIDI_SOURCE_COMMIT"] = source_metadata["commit"]
else:
    for attempt in range(1, 4):
        try:
            subprocess.run([
                "git", "clone", "--branch", BRANCH, "--single-branch",
                REPO_URL, str(workspace),
            ], check=True)
            break
        except subprocess.CalledProcessError:
            if attempt == 3:
                raise
            if workspace.exists():
                shutil.rmtree(workspace)
            time.sleep(15 * attempt)
required_imports = ["numpy", "scipy", "soundfile", "pandas", "librosa", "matplotlib", "tqdm", "yaml", "tensorflow"]
versions = {}
for module_name in required_imports:
    module = importlib.import_module(module_name)
    versions[module_name] = getattr(module, "__version__", "available")
print(json.dumps({"runtime": sys.version, "packages": versions}, indent=2))


In [ ]:
command = [
    sys.executable,
    str(workspace / "scripts/cloud/kaggle_entrypoint.py"),
    "--task", TASK,
    "--input-root", "/kaggle/input",
    "--workers", str(WORKERS),
]
subprocess.run(command, cwd=workspace, check=True)
subprocess.run([
    sys.executable,
    str(workspace / "scripts/cloud/package_kaggle_outputs.py"),
    "--task", TASK,
    "--output-dir", "/kaggle/working/guitar-midi-results",
], cwd=workspace, check=True)


In [ ]:
results = pathlib.Path("/kaggle/working/guitar-midi-results")
materialized_input = pathlib.Path("/kaggle/working/guitar-midi-input")
if materialized_input.exists():
    shutil.rmtree(materialized_input)
if workspace.exists():
    shutil.rmtree(workspace)
for path in sorted(results.iterdir()):
    print(f"{path.name}: {path.stat().st_size} bytes")
